# Knowledge Graph Explorer — Navegador

Use this notebook to inspect and validate the survey ontology after running
`scripts/setup/bootstrap_kg_ontology.py`.

**Workflow:**
1. Run Cell 1 (load KG + stats)
2. Browse constructs per topic (Cell 3)
3. Test comparability of specific question pairs (Cell 4)
4. Review cross-domain relationships (Cell 5)
5. Edit `data/kg_ontology.json` as needed, then reload (Cell 6)

In [ ]:
# Cell 1 — Load KG and print summary stats
import sys
sys.path.insert(0, '..')  # add project root to path

from survey_kg import kg, _KG_AVAILABLE

if not _KG_AVAILABLE or kg is None:
    print('KG not available. Run: python scripts/setup/bootstrap_kg_ontology.py')
else:
    stats = kg.summary_stats()
    print('=== Knowledge Graph Summary ===')
    for k, v in stats.items():
        print(f'  {k:20s}: {v}')
    print()
    print('KG is ready for exploration.')

In [ ]:
# Cell 2 — Visualise the ontology graph
# Shows domains and constructs as a node-edge diagram (questions omitted for clarity)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

if kg is None:
    print('KG not loaded.')
else:
    # Build a subgraph with only domain and construct nodes
    sub_nodes = [
        n for n, d in kg.G.nodes(data=True)
        if d.get('node_type') in ('domain', 'construct')
    ]
    sub = kg.G.subgraph(sub_nodes)

    color_map = []
    for n in sub.nodes():
        ntype = kg.G.nodes[n].get('node_type')
        color_map.append('#4A90D9' if ntype == 'domain' else '#F5A623')

    labels = {n: kg.G.nodes[n].get('label', n)[:20] for n in sub.nodes()}

    fig, ax = plt.subplots(figsize=(18, 12))
    pos = nx.spring_layout(sub, k=1.5, seed=42)
    nx.draw_networkx(
        sub, pos=pos, labels=labels, node_color=color_map,
        node_size=800, font_size=7, ax=ax, arrows=True,
        edge_color='#cccccc', width=0.8
    )
    legend = [
        mpatches.Patch(color='#4A90D9', label='Domain'),
        mpatches.Patch(color='#F5A623', label='Construct'),
    ]
    ax.legend(handles=legend, loc='upper left')
    ax.set_title('Survey Ontology — Domains and Constructs', fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 3 — Browse by topic
# Change TOPIC below to any domain abbreviation (e.g. 'IDE', 'POB', 'SAL')

TOPIC = 'IDE'   # <-- change this

if kg is None:
    print('KG not loaded.')
else:
    from dataset_knowledge import rev_topic_dict
    label = rev_topic_dict.get(TOPIC, TOPIC)
    print(f'=== Topic: {TOPIC} — {label} ===\n')

    questions = kg.get_topic_questions(TOPIC)
    if not questions:
        print(f'No questions found for topic {TOPIC}. Is the ontology loaded?')
    else:
        # Group by construct
        by_construct = {}
        for qid, info in questions.items():
            c = info.get('construct') or 'Unmapped'
            by_construct.setdefault(c, []).append((qid, info.get('text', '')))

        for construct_id, qs in sorted(by_construct.items()):
            construct_label = kg.G.nodes.get(construct_id, {}).get('label', construct_id)
            print(f'  [{construct_id}] {construct_label} ({len(qs)} questions)')
            for qid, text in qs[:5]:
                if '|' in text:
                    text = text.split('|', 1)[-1]
                print(f'    • {qid}: {text[:100]}')
            if len(qs) > 5:
                print(f'    ... (+{len(qs)-5} more)')
            print()

In [ ]:
# Cell 4 — Test comparability between any two question IDs
# Paste real QIDs from the survey (format: pN_M|TOPIC)

Q1 = 'p5_1|IDE'   # <-- change this
Q2 = 'p3_2|POB'   # <-- change this

if kg is None:
    print('KG not loaded.')
else:
    result = kg.are_comparable(Q1, Q2)
    symbol = '✓' if result['comparable'] is True else ('~' if result['comparable'] == 'weak' else '✗')
    print(f'{symbol}  {Q1} ↔ {Q2}')
    print(f'   Comparable : {result["comparable"]}')
    print(f'   Type       : {result["comparison_type"]}')
    print(f'   Reason     : {result["reason"]}')
    if 'caution' in result:
        print(f'   ⚠  Caution  : {result["caution"]}')
    print()
    print('--- Full prompt context block ---')
    print(kg.get_kg_context_for_prompt([Q1, Q2]))

In [ ]:
# Cell 5 — Cross-domain relationships table
import pandas as pd

if kg is None:
    print('KG not loaded.')
else:
    rels = kg.get_cross_domain_relationships()
    if not rels:
        print('No cross-domain relationships in the graph yet.')
        print('Run bootstrap_kg_ontology.py to generate them.')
    else:
        df = pd.DataFrame(rels)[['from', 'from_domain', 'relation', 'to', 'to_domain', 'strength', 'evidence']]
        print(f'{len(df)} cross-domain relationships:\n')
        pd.set_option('display.max_colwidth', 60)
        pd.set_option('display.max_rows', 50)
        display(df.sort_values('strength', ascending=False))

In [ ]:
# Cell 6 — Reload the KG after editing kg_ontology.json
from pathlib import Path
import importlib, survey_kg as _skg

ONTOLOGY_PATH = Path('..') / 'data' / 'kg_ontology.json'

if not ONTOLOGY_PATH.exists():
    print(f'File not found: {ONTOLOGY_PATH}')
else:
    kg.load_from_json(ONTOLOGY_PATH)
    stats = kg.summary_stats()
    print('KG reloaded.')
    for k, v in stats.items():
        print(f'  {k:20s}: {v}')